# Hardware Test in Pynq-Z2 Board

In [85]:
from pathlib import Path
from typing import Tuple

import numpy as np

from axi_stream_driver import NeuralNetworkOverlay

## Data

In [86]:
test_data = np.load("test_raw.npz", allow_pickle=True)

X_test = test_data["data"]
y_test_label = test_data["labels"]

label_map = {"lateral_L": 0, "lateral_R": 1, "supine": 2}
y_test = np.vectorize(label_map.get)(y_test_label)

print("\nDataset shapes:")
print(f"  Test:  {X_test.shape}, labels: {y_test.shape}")


Dataset shapes:
  Test:  (120001, 64), labels: (120001,)


In [87]:
SAMPLES = 1000

# Set random seed for reproducibility
np.random.seed(42)
random_indices = np.random.choice(len(X_test), size=SAMPLES, replace=False)

# X_test = X_test[random_indices]
# y_test = y_test[random_indices]

### Quantization

In [88]:
# ap_ufixed<W,I>

INPUT_WORD = 8.0
INPUT_INT = 8.0
INPUT_SCALE = 2.0 ** (INPUT_WORD - INPUT_INT)
print(INPUT_SCALE)

# ap_fixed<W,I>

OUTPUT_WORD = 16.0
OUTPUT_INT = 8.0
OUTPUT_SCALE = 2.0 ** (OUTPUT_WORD - OUTPUT_INT)
print(OUTPUT_SCALE)

encode_v = lambda xi: np.uint8(np.round(xi * INPUT_SCALE))
decode_v = lambda yi: np.float32(yi) / OUTPUT_SCALE

1.0
256.0


## Neural Network

In [89]:
# Instantiate overlay for single or batch inference (64 in, 3 out)
nn = NeuralNetworkOverlay(
    "myproject.bit",
    x_shape=(64,),
    y_shape=(3,),
    input_dtype=np.uint16,   # 16-bit AXI-Stream for ap_ufixed<7,8>
    output_dtype=np.int32,   # 32-bit AXI-Stream for ap_fixed<14,9>
)

# Run inference on the full test set in one call
y_hw, dts, rate = nn.predict(
    X_test, profile=True, encode=encode_v, decode=decode_v
)

HW inference: 100%|██████████| 120001/120001 [02:15<00:00, 882.73it/s]


Classified 120001 samples in 152.515162 seconds (786.81 inferences / s)


### Evaluate

In [90]:
# Compute predicted classes from raw logits
y_pred_hw = np.argmax(y_hw, axis=1)

accuracy = np.mean(y_pred_hw == y_test)
print(f"Hardware accuracy: {accuracy*100:.2f}%")
print(f"\nFirst 10 logits:\n{y_hw[:10]}")
print(f"\nFirst 10 predictions:   {y_pred_hw[:10]}")
print(f"First 10 ground truth:  {y_test[:10]}")

Hardware accuracy: 86.38%

First 10 logits:
[[ 3.3242188   1.828125   -1.6875    ]
 [ 3.890625   -4.2304688  -0.52734375]
 [-3.9375      0.78125     2.71875   ]
 [-2.03125     6.1210938  -0.3828125 ]
 [-1.3710938   2.5585938   1.109375  ]
 [-1.2070312   0.2265625   0.9921875 ]
 [-1.0585938   2.484375    0.03515625]
 [ 0.72265625 -2.78125     1.8789062 ]
 [-1.4375      1.6914062   1.7070312 ]
 [ 2.9726562  -3.5390625   0.54296875]]

First 10 predictions:   [0 0 2 1 1 2 1 2 2 0]
First 10 ground truth:  [0 0 2 1 1 2 1 0 1 0]


## Save

In [91]:
np.savez("y_hw.npz", logits=y_hw, predictions=y_pred_hw)